# Data Governance - Phase 1

# Libraries

In [ ]:
!pip install great_expectations==0.18.21
!pip install sweetviz

In [ ]:
import pandas as pd
import numpy as np
import sweetviz as sv
import re
import great_expectations as gx

---
#  Data Profiling

In [ ]:
df = pd.read_csv("new_retail_data.csv")
df.info()

In [ ]:
print("Shape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nSummary Statistics:\n", df.describe())

In [ ]:
report = sv.analyze(df)
report.show_html('sweetviz_report.html')
print("Profiling report saved as sweetviz_report.html")

---
#  Data Cleaning

Cleaning order:
1. Cast columns to correct data types
2. Remove duplicate rows
3. Fill missing values
4. Standardize text columns
5. Clean phone numbers and emails
6. Fix logical inconsistencies

In [ ]:
for col in ['Transaction_ID', 'Customer_ID', 'Zipcode']:
    df[col] = df[col].astype(str).str.replace('.0', '', regex=False).str.strip()
    df[col] = df[col].replace({'nan': np.nan, 'None': np.nan, '': np.nan})

df['Phone'] = df['Phone'].astype(str).str.strip()
df['Phone'] = df['Phone'].replace({'nan': np.nan, 'None': np.nan, '': np.nan})

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

date_mode = df['Date'].mode()[0]
df['Date'] = df['Date'].fillna(date_mode)

df['Month'] = df['Date'].dt.month_name().str.lower()
df['Year'] = df['Date'].dt.year

num_cols = ['Age', 'Total_Purchases', 'Amount', 'Ratings', 'Year']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Data types cast correctly")

In [ ]:
before_rows = df.shape[0]
df = df.drop_duplicates()
after_rows = df.shape[0]
print(f"Removed {before_rows - after_rows} duplicate rows")
print(f"Rows remaining: {after_rows}")

In [ ]:
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

mode_cat_cols = [
    'City', 'State', 'Country', 'Gender',
    'Income', 'Customer_Segment', 'Month', 'Product_Category',
    'Product_Brand', 'Product_Type', 'Feedback', 'Shipping_Method',
    'Payment_Method', 'Order_Status', 'products'
]
for col in mode_cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

df['Name'] = df['Name'].fillna('unknown')
df['Address'] = df['Address'].fillna('unknown')

df['Transaction_ID'] = df['Transaction_ID'].fillna('UNKNOWN')
df['Customer_ID'] = df['Customer_ID'].fillna('UNKNOWN')

df['Zipcode'] = df['Zipcode'].astype(str).str.replace('.0', '', regex=False).str.strip()
df['Zipcode'] = df['Zipcode'].replace({'nan': np.nan, 'None': np.nan, '': np.nan})
df['Zipcode'] = df['Zipcode'].fillna('UNKNOWN')

df['Time'] = df['Time'].fillna(df['Time'].mode()[0])

df['Email'] = df['Email'].fillna('unknown')

print("Missing values filled")
print("Remaining missing values:\n", df.isnull().sum())

In [ ]:
text_cols = [
    'Name', 'City', 'State', 'Country', 'Gender',
    'Product_Category', 'Product_Brand', 'Product_Type',
    'Income', 'Customer_Segment', 'Feedback', 'Shipping_Method',
    'Payment_Method', 'Order_Status', 'products', 'Address'
]
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()
    df[col] = df[col].replace('nan', 'unknown')

df['Gender'] = df['Gender'].replace({'m': 'male', 'f': 'female'})

print("Text columns standardized")

In [ ]:
def clean_phone(phone):
    if pd.isnull(phone):
        return 'UNKNOWN'
    phone = str(phone)
    phone = re.sub(r'[^0-9]', '', phone)
    if phone == '':
        return 'UNKNOWN'
    return phone

df['Phone'] = df['Phone'].apply(clean_phone)
print("Phone cleaned. Sample:", df['Phone'].head(3).tolist())

In [ ]:
def clean_email(email):
    if pd.isnull(email) or str(email).strip() == '':
        return 'unknown'
    email = str(email).strip().lower()
    if email in ('unknown', 'nan', 'none'):
        return 'unknown'
    pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$'
    if re.match(pattern, email):
        return email
    return 'invalid_email'

df['Email'] = df['Email'].apply(clean_email)
print("Email value counts:")
print(df['Email'].value_counts().head(5))

In [ ]:
df['Age'] = df['Age'].clip(18, 70)
df['Month'] = np.where(df['Date'].notna(), df['Date'].dt.month_name().str.lower(), df['Month'])
df['Year'] = np.where(df['Date'].notna(), df['Date'].dt.year, df['Year'])
df['Total_Amount'] = (df['Amount'] * df['Total_Purchases']).round(2)
print("Logical inconsistencies fixed")

In [ ]:
print("=== Cleaning Summary ===")
print("Final Shape:", df.shape)
print("\nMissing Values after cleaning:\n", df.isnull().sum())
df.info()

In [ ]:
df.to_csv("cleaned_dataset.csv", index=False)
print("Cleaning completed successfully")
print("Saved as: cleaned_dataset.csv")

---
#  Data Validation


In [ ]:
context = gx.get_context()

datasource_name = "my_pandas_datasource"
asset_name = "cleaned_data_asset"
suite_name = "project_suite"

try:
    datasource = context.sources.add_pandas(datasource_name)
except:
    datasource = context.sources.get(datasource_name)

try:
    asset = datasource.add_dataframe_asset(name=asset_name)
except:
    asset = datasource.get_asset(asset_name)

context.add_or_update_expectation_suite(expectation_suite_name=suite_name)

batch_request = asset.build_batch_request(dataframe=df)

validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=suite_name
)

print("Validator created successfully")

In [ ]:
validator.expect_column_values_to_not_be_null("Transaction_ID")
validator.expect_column_values_to_not_be_null("Customer_ID")
validator.expect_column_values_to_not_be_null("Amount")
validator.expect_column_values_to_not_be_null("Total_Purchases")
validator.expect_column_values_to_not_be_null("Total_Amount")
validator.expect_column_values_to_not_be_null("Email")
validator.expect_column_values_to_not_be_null("Phone")

print("Not-null expectations set")

In [ ]:
validator.expect_column_values_to_be_between("Age", min_value=18, max_value=70)
validator.expect_column_values_to_be_between("Amount", min_value=0)
validator.expect_column_values_to_be_between("Total_Purchases", min_value=1)
validator.expect_column_values_to_be_between("Total_Amount", min_value=0)
validator.expect_column_values_to_be_between("Ratings", min_value=1, max_value=5)

print("Range expectations set")

In [ ]:
validator.expect_column_values_to_be_in_set(
    "Gender",
    value_set=["male", "female", "unknown"]
)

validator.expect_column_values_to_be_in_set(
    "Order_Status",
    value_set=["pending", "processing", "shipped", "delivered", "unknown"]
)

validator.expect_column_values_to_be_in_set(
    "Shipping_Method",
    value_set=["standard", "express", "same-day", "unknown"]
)

validator.expect_column_values_to_be_in_set(
    "Payment_Method",
    value_set=["cash", "credit card", "debit card", "paypal", "unknown"]
)

print("Value set expectations set")

In [ ]:
validator.expect_column_values_to_match_regex(
    "Email",
    r"(^[\w\.-]+@[\w\.-]+\.\w+$)|(^invalid_email$)|(^unknown$)"
)

validator.expect_column_values_to_match_regex(
    "Phone",
    r"(^[0-9]+$)|(^UNKNOWN$)"
)

print("Regex expectations set")

In [ ]:
non_unknown_ids = df[df["Transaction_ID"] != "UNKNOWN"]["Transaction_ID"]
dup_count = non_unknown_ids.duplicated().sum()
print("Duplicate Transaction_ID values (excluding UNKNOWN):", dup_count)

In [ ]:
wrong_total = (df["Total_Amount"].round(2) != (df["Amount"] * df["Total_Purchases"]).round(2)).sum()
print("Rows with incorrect Total_Amount:", wrong_total)

In [ ]:
name_conflicts  = df.groupby("Customer_ID")["Name"].nunique()
email_conflicts = df.groupby("Customer_ID")["Email"].nunique()
phone_conflicts = df.groupby("Customer_ID")["Phone"].nunique()

print("Customer_IDs linked to more than one Name :", (name_conflicts  > 1).sum())
print("Customer_IDs linked to more than one Email:", (email_conflicts > 1).sum())
print("Customer_IDs linked to more than one Phone:", (phone_conflicts > 1).sum())

In [ ]:
validator.save_expectation_suite(discard_failed_expectations=False)
print("Expectation suite saved")

In [ ]:
results = validator.validate()
print("Validation success:", results["success"])

In [ ]:
print("Validation Statistics:")
print(results["statistics"])
num_passed = results["statistics"]["successful_expectations"]
num_failed = results["statistics"]["unsuccessful_expectations"]
print(f"\nPassed: {num_passed}")
print(f"Failed: {num_failed}")

In [ ]:
print("=== Per-Expectation Results ===")
for result in results["results"]:
    exp_type = result["expectation_config"]["expectation_type"]
    col = result["expectation_config"]["kwargs"].get("column", "")
    status = "PASSED" if result["success"] else "FAILED"
    print(f"{status}: {exp_type} | column: {col}")

# PHASE 2

# Imports and Load Dataset

In [ ]:
import pandas as pd
import hashlib
from datetime import datetime

In [ ]:
df = pd.read_csv("cleaned_dataset.csv")
df.head()

# HASHING

In [ ]:
sensitive_columns = [
    "Transaction_ID",
    "Customer_ID",
    "Name",
    "Email",
    "Phone",
    "Address"
]

sensitive_columns = [col for col in sensitive_columns if col in df.columns]

print("Sensitive columns:", sensitive_columns)

# Hash function
def hash_value(value):
    value = str(value)
    return hashlib.sha256(value.encode()).hexdigest()

# Apply Hashing
for col in sensitive_columns:
    df[col + "_hashed"] = df[col].apply(hash_value)
hashed_columns = [col + "_hashed" for col in sensitive_columns]

hashing_output_columns = []
for col in sensitive_columns:
    hashing_output_columns.append(col)
    hashing_output_columns.append(col + "_hashed")

df[hashing_output_columns].head()

# RSA Encryption and Decryption

In [ ]:
# GCD
def gcd(a, b):
    while b != 0:
        a, b = b, a % b
    return a

# Modular inverse
def mod_inverse(e, phi):
    for d in range(1, phi):
        if (d * e) % phi == 1:
            return d
    return None

# Generate RSA
def generate_rsa_keys():
    p = 61
    q = 53

    n = p * q
    phi = (p - 1) * (q - 1)

    e = 17
    d = mod_inverse(e, phi)

    public_key = (e, n)
    private_key = (d, n)

    return public_key, private_key

In [ ]:
public_key, private_key = generate_rsa_keys()

print("Public Key:", public_key)
print("Private Key:", private_key)

In [ ]:
# Encrypt
def rsa_encrypt(text, public_key):
    e, n = public_key
    encrypted_text = []

    for char in str(text):
        encrypted_text.append(pow(ord(char), e, n))
    return encrypted_text

# Decrypt
def rsa_decrypt(encrypted_text, private_key):
    d, n = private_key
    decrypted_text = ""

    for number in encrypted_text:
        decrypted_text += chr(pow(number, d, n))
    return decrypted_text

In [ ]:
df["Encrypted_Product_Type"] = df["Product_Type"].apply(
    lambda x: rsa_encrypt(x, public_key)
)

df["Decrypted_Product_Type"] = df["Encrypted_Product_Type"].apply(
    lambda x: rsa_decrypt(x, private_key)
)

print("RSA Encryption/Decryption applied successfully")
print("Sample verification:")

for i in range(3):
    original = df["Product_Type"].iloc[i]
    decrypted = df["Decrypted_Product_Type"].iloc[i]
    match = "MATCH" if original == decrypted else "MISMATCH"
    print(f"Row {i}: {original} -> {match}")

In [ ]:
rsa_result = pd.DataFrame({
    "Plain Text": df["Product_Type"].head(10),
    "Encrypted Text": df["Encrypted_Product_Type"].head(10),
    "Decrypted Text": df["Decrypted_Product_Type"].head(10)
})

rsa_result

In [ ]:
rsa_result.to_csv("rsa_encryption_decryption_output.csv", index=False)
print("RSA output saved as rsa_encryption_decryption_output.csv")

# Masking

In [ ]:
def mask_phone(phone):
    phone = str(phone)

    if len(phone) <= 4:
        return phone

    return "*" * (len(phone) - 4) + phone[-4:]

In [ ]:
def mask_email(email):
    email = str(email)

    if "@" not in email:
        return email

    name_part, domain_part = email.split("@", 1)

    if len(name_part) <= 2:
        masked_name = name_part[0] + "*"
    else:
        masked_name = name_part[0] + "*" * (len(name_part) - 2) + name_part[-1]

    return masked_name + "@" + domain_part

In [ ]:
if "Phone" in df.columns:
    df["Phone_masked"] = df["Phone"].apply(mask_phone)

if "Email" in df.columns:
    df["Email_masked"] = df["Email"].apply(mask_email)

masking_columns = []

if "Phone" in df.columns:
    masking_columns += ["Phone", "Phone_masked"]

if "Email" in df.columns:
    masking_columns += ["Email", "Email_masked"]

df[masking_columns].head()

# ABAC

In [ ]:
COLUMN_SENSITIVITY = {
    "Name": "HIGH",
    "Email": "HIGH",
    "Phone": "HIGH",
    "Address": "HIGH",
    "Zipcode": "HIGH",
    "Customer_ID": "HIGH",
    "Transaction_ID": "HIGH",
    "Income": "HIGH",
    "Amount": "HIGH",
    "Total_Amount": "HIGH",

    "Age": "MEDIUM",
    "Gender": "MEDIUM",
    "Customer_Segment": "MEDIUM",
    "Total_Purchases": "MEDIUM",
    "Payment_Method": "MEDIUM",
    "Order_Status": "MEDIUM",
    "Ratings": "MEDIUM",
    "Feedback": "MEDIUM",
    "Date": "MEDIUM",
    "Year": "MEDIUM",
    "Month": "MEDIUM",
    "Time": "MEDIUM",

    "City": "LOW",
    "State": "LOW",
    "Country": "LOW",
    "Product_Category": "LOW",
    "Product_Brand": "LOW",
    "Product_Type": "LOW",
    "Shipping_Method": "LOW",
    "products": "LOW",
}

In [ ]:
USERS = {
    "admin_sara": {
        "role": "Admin",
        "department": "IT",
        "clearance": 3,
        "can_write": True,
        "active_hours": (0, 23),
    },

    "analyst_omar": {
        "role": "Data Analyst",
        "department": "Analytics",
        "clearance": 2,
        "can_write": False,
        "active_hours": (8, 18),
    },

    "manager_lina": {
        "role": "Manager",
        "department": "Sales",
        "clearance": 2,
        "can_write": True,
        "active_hours": (8, 20),
    },

    "intern_ahmed": {
        "role": "Intern",
        "department": "Marketing",
        "clearance": 1,
        "can_write": False,
        "active_hours": (9, 17),
    },

    "auditor_nour": {
        "role": "Auditor",
        "department": "Compliance",
        "clearance": 3,
        "can_write": False,
        "active_hours": (0, 23),
    },
}

In [ ]:
POLICIES = {
    "READ": {
        "LOW": 1,
        "MEDIUM": 2,
        "HIGH": 3,
    },

    "WRITE": {
        "LOW": 2,
        "MEDIUM": 2,
        "HIGH": 3,
    },
}

In [ ]:
class ABACEngine:

    def __init__(self, users, policies, sensitivity):
        self.users = users
        self.policies = policies
        self.sensitivity = sensitivity
        self.access_log = []

    def _log(self, username, action, columns, granted, reason):
        entry = {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "username": username,
            "action": action,
            "columns": columns,
            "granted": granted,
            "reason": reason,
        }

        self.access_log.append(entry)

        status = "GRANTED" if granted else "DENIED"

        print(
            f"[{entry['timestamp']}] {status} | "
            f"user={username} | action={action} | reason={reason}"
        )

    def _get_user(self, username):
        user = self.users.get(username)

        if not user:
            raise ValueError(f"Unknown user: {username}")

        return user

    def _is_within_active_hours(self, user):
        current_hour = datetime.now().hour
        start, end = user["active_hours"]

        return start <= current_hour <= end

    def evaluate(self, username, action, columns):

        action = action.upper()

        if action not in ("READ", "WRITE"):
            raise ValueError("Action must be READ or WRITE")

        user = self._get_user(username)

        if not self._is_within_active_hours(user):

            self._log(
                username,
                action,
                columns,
                False,
                f"Access outside allowed hours "
                f"{user['active_hours'][0]}-{user['active_hours'][1]}"
            )

            return {
                "granted_columns": [],
                "denied_columns": columns,
                "overall_granted": False
            }

        if action == "WRITE" and not user["can_write"]:

            self._log(
                username,
                action,
                columns,
                False,
                "User cannot perform WRITE operation"
            )

            return {
                "granted_columns": [],
                "denied_columns": columns,
                "overall_granted": False
            }

        granted_cols = []
        denied_cols = []

        for col in columns:

            sensitivity = self.sensitivity.get(col, "HIGH")

            required = self.policies[action].get(sensitivity, 1)

            if user["clearance"] >= required:

                granted_cols.append(col)

            else:

                denied_cols.append(col)

                self._log(
                    username,
                    action,
                    [col],
                    False,
                    f"{col} requires clearance {required}"
                )

        if granted_cols:

            self._log(
                username,
                action,
                granted_cols,
                True,
                f"Clearance {user['clearance']} accepted"
            )

        overall = len(denied_cols) == 0

        return {
            "granted_columns": granted_cols,
            "denied_columns": denied_cols,
            "overall_granted": overall
        }

    def get_audit_log(self):
        return pd.DataFrame(self.access_log)

In [ ]:
class SecureDataAccessor:

    def __init__(self, dataframe, engine):
        self._df = dataframe
        self._engine = engine

    def read(self, username, columns, nrows=5):

        print(f"READ REQUEST | User: {username}")
        print(f"Columns: {columns}")


        result = self._engine.evaluate(username, "READ", columns)

        if not result["granted_columns"]:

            print("No accessible columns.\n")

            return None

        if result["denied_columns"]:

            print(
                f"Hidden columns: "
                f"{result['denied_columns']}"
            )

        df_view = self._df[result["granted_columns"]].head(nrows)

        print("\nReturned Data:\n")

        print(df_view.to_string(index=False))

        return df_view

    def write(self, username, column, row_index, value):

        print(f"WRITE REQUEST | User: {username}")
        print(f"Column: {column}")
        print(f"Row: {row_index}")
        print(f"Value: {value}")

        result = self._engine.evaluate(
            username,
            "WRITE",
            [column]
        )

        if result["overall_granted"]:

            self._df.at[row_index, column] = value

            print("Write successful.\n")

        else:

            print("Write denied.\n")

In [ ]:
engine = ABACEngine(
    USERS,
    POLICIES,
    COLUMN_SENSITIVITY
)

accessor = SecureDataAccessor(
    df,
    engine
)

In [ ]:
user_rows = []

for uname, attrs in USERS.items():

    user_rows.append({
        "Username": uname,
        "Role": attrs["role"],
        "Department": attrs["department"],
        "Clearance": attrs["clearance"],
        "Can Write": attrs["can_write"],
        "Hours": f"{attrs['active_hours'][0]}-{attrs['active_hours'][1]}"
    })

users_df = pd.DataFrame(user_rows)

print(users_df.to_string(index=False))

In [ ]:
sens_rows = []

for col, level in COLUMN_SENSITIVITY.items():

    sens_rows.append({
        "Column": col,
        "Sensitivity": level
    })

sens_df = pd.DataFrame(sens_rows)

print(sens_df.to_string(index=False))

In [ ]:
pol_rows = []

for action, levels in POLICIES.items():

    for sensitivity, clearance in levels.items():

        pol_rows.append({
            "Action": action,
            "Sensitivity": sensitivity,
            "Min Clearance": clearance
        })

pol_df = pd.DataFrame(pol_rows)

print(pol_df.to_string(index=False))

In [ ]:
high_cols = [
    "Name",
    "Email",
    "Phone",
    "Transaction_ID",
    "Amount"
]

accessor.read(
    "admin_sara",
    high_cols,
    nrows=3
)

In [ ]:
mixed_cols = [
    "Name",
    "Email",
    "City",
    "Product_Category",
    "Ratings"
]

accessor.read(
    "analyst_omar",
    mixed_cols,
    nrows=3
)

In [ ]:
low_cols = [
    "City",
    "Country",
    "Product_Brand",
    "Shipping_Method"
]

accessor.read(
    "intern_ahmed",
    low_cols,
    nrows=3
)

In [ ]:
accessor.write(
    "intern_ahmed",
    "City",
    0,
    "Cairo"
)

In [ ]:
accessor.write(
    "manager_lina",
    "Order_Status",
    0,
    "Delivered"
)

In [ ]:
all_high = [
    "Name",
    "Email",
    "Phone",
    "Income",
    "Total_Amount"
]

accessor.read(
    "auditor_nour",
    all_high,
    nrows=3
)

In [ ]:
accessor.write(
    "auditor_nour",
    "Income",
    0,
    "High"
)

In [ ]:
log_df = engine.get_audit_log()

print(
    log_df[
        [
            "timestamp",
            "username",
            "action",
            "granted",
            "reason"
        ]
    ].to_string(index=False)
)

# **Save Protected Datset**

In [ ]:
df.to_csv("protected_dataset.csv", index=False)
print("Protected dataset saved as protected_dataset.csv")

## Bonus 1: RSA Attack

In this bonus part, a simple factorization attack is applied on RSA.  
The attacker only knows the public key `(e, n)` and tries to factorize `n` into `p` and `q`.  
After finding `p` and `q`, the attacker recomputes `phi`, recovers the private key `d`, and decrypts the ciphertext again.

This attack works here because the RSA keys used in this project are small for educational purposes.  
In real RSA systems, very large prime numbers are used, so this attack becomes computationally infeasible.

In [ ]:
# RSA attack using factorization

e, n = public_key

def factorize_n(n):
    for i in range(2, n):
        if n % i == 0:
            return i, n // i
    return None, None


def rsa_attack_decrypt(encrypted_text, d, n):
    decrypted_text = ""

    for number in encrypted_text:
        decrypted_text += chr(pow(number, d, n))

    return decrypted_text


attacker_p, attacker_q = factorize_n(n)

print("Attacker found p =", attacker_p)
print("Attacker found q =", attacker_q)

attacker_phi = (attacker_p - 1) * (attacker_q - 1)
attacker_d = mod_inverse(e, attacker_phi)

print("Recovered private key d =", attacker_d)

df["Attacked_Decrypted_Product_Type"] = df["Encrypted_Product_Type"].apply(
    lambda x: rsa_attack_decrypt(x, attacker_d, n)
)

rsa_attack_result = pd.DataFrame({
    "Original Plain Text": df["Product_Type"].head(10),
    "Cipher Text": df["Encrypted_Product_Type"].head(10),
    "Normal Decrypted Text": df["Decrypted_Product_Type"].head(10),
    "Attack Decrypted Text": df["Attacked_Decrypted_Product_Type"].head(10)
})

rsa_attack_result["Same Result"] = (
    rsa_attack_result["Original Plain Text"] == rsa_attack_result["Attack Decrypted Text"]
)

rsa_attack_result

In [ ]:
rsa_attack_result.to_csv("rsa_attack_output.csv", index=False)
print("RSA attack output saved successfully")

## Bonus 2: Machine Learning Preprocessing and Model

In this bonus part, the dataset is prepared to be used as input for a machine learning model.

The goal of the model is to predict whether a customer is a high value customer or not.  
The target column `High_Value_Customer` is created using the median of `Total_Amount`.

To avoid data leakage, `Total_Amount` and `Amount` are not used as input features because they are directly related to the target.  
Also, the train/test split is done before scaling, so the scaler is fitted only on the training data.

Categorical columns are encoded, numerical columns are scaled, and a Decision Tree Classifier is trained and evaluated using accuracy and classification report

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

df_ml = df.copy()

df_ml["High_Value_Customer"] = (
    df_ml["Total_Amount"] > df_ml["Total_Amount"].median()
).astype(int)

features = [
    "Age",
    "Total_Purchases",
    "Ratings",
    "Product_Category",
    "Product_Type",
    "Gender"
]

df_ml = df_ml[features + ["High_Value_Customer"]].copy()

X = df_ml[features]
y = df_ml["High_Value_Customer"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train_encoded = pd.get_dummies(X_train)
X_test_encoded = pd.get_dummies(X_test)

X_test_encoded = X_test_encoded.reindex(
    columns=X_train_encoded.columns,
    fill_value=0
)

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))